# Jinja

With the Jinja templating language, you can parametrise and customise your dbt models, as well as avoiding a lot of boilerplate by defining the marcores that substitute the repeating construction automatically.

Jinja is critically important for dbt, as some of the tools defined by the dbt are ethe core functionality of the framework. For example, the `ref` macro is one such tool.

## Reference

You can invoke one dbt object from another. This can be achieved using the `ref` function. Where the SQL syntax requires the data source, you can use the Jinja template `{{ ref('name') }}`. The dbt will generate the corresponding dialect specific sql code.

---

The following cells define the project and the `model_a.sql` file we will reference as an example. 

In [11]:
#init
dbt init knowledge --profile knowledge -q
cd knowledge

In [31]:
#file knowledge/models/model_a.sql
SELECT * FROM (
VALUES
    ('hello', 1),
    ('world', 2)
) AS tab(col1, col2)

The following cell generates the `model_b.sql` that uses `ref('model_a')` as the datasource.

In [33]:
#file knowledge/models/model_b.sql
SELECT * FROM {{ ref('model_a') }} WHERE col2=2

The data generated by `model_b` corresponds to a subset of the `model_a`.

In [35]:
dbt show -q --select model_b.sql

| col1  | col2 |
| ----- | ---- |
| world |    2 |



### Folders

It can sometimes be confusing how to reference a model that is defined in a subfolder of `models`. As dbt project require unique model names, simply pass the model name to the `ref` macro.

---

The following cells define the dbt project with the `models/staging/example_staging.sql` model.

In [1]:
#init
dbt init knowledge --profile knowledge -q
cd knowledge
mkdir models/staging

In [2]:
#file knowledge/models/staging/example_staging.sql
SELECT 'select from staging' as info

The example model refering to the `exapmle_staging` model:

In [3]:
#file knowledge/models/dest.sql
SELECT * FROM {{ ref('example_staging') }}

As the result `dest` correctly uses information from the `example_staging`:

In [4]:
dbt run -q
dbt show --select dest -q

| info                |
| ------------------- |
| select from staging |



### Packages

Another common scenario is when the model reference other packages. The syntax in such case is `ref('<package_name>', '<model_name>')`.

---

The following cell configures the project that uses the `dbt_project_evaluator` package, which contains some models that we will demonstrate how to references from our models.

In [5]:
#init
dbt init knowledge --profile knowledge -q
cd knowledge

cat << EOF >> packages.yml
packages:
  - package: dbt-labs/dbt_project_evaluator
    version: 1.3.0
EOF

dbt deps -q

This is an example of a model that references a model from the another package:

In [8]:
#file knowledge/models/my_model.sql
SELECT parent_id, parent FROM {{ ref('dbt_project_evaluator', 'int_all_dag_relationships') }}

And the demonstration that model really retrieves the model from the package:

In [9]:
dbt show --select my_model -q

| parent_id            | parent              |
| -------------------- | ------------------- |
| model.knowledge.m... | my_first_dbt_model  |
| model.knowledge.m... | my_first_dbt_model  |
| model.knowledge.m... | my_second_dbt_model |



## Macros

Macros allow you to create a reusable pieces of Jinja code. These can then be invoked from different parts of the DBT project.

---

In [1]:
#init
dbt init macros --profile knowledge -q
cd macros

The following cell defines a simple macro that repeats the `hello world` pattern a set number of times:

In [9]:
# file macros/macros/my_macro.sql
{% macro hello_world(number) %}
    {% for i in range(number) %}hello world
    {% endfor %}
{% endmacro %}

The model that invokes this macros:

In [10]:
# file macros/models/test_model.sql
{{ hello_world(5) }}

The compilation result:

In [12]:
dbt compile --select test_model -q


    hello world
    hello world
    hello world
    hello world
    hello world
    

